In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU PRESENT")

In [ ]:
# podaci
!mkdir -p data && cd data && \
  curl -sL -O https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz && \
  curl -sL -O https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat && \
  tar -xzf 102flowers.tgz
!ls data/jpg | wc -l

In [ ]:
import os, json, time, numpy as np, pandas as pd, scipy.io
import torch, torch.nn as nn
from pathlib import Path
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("assets/tmp", exist_ok=True)
print(device)

In [ ]:
DROP_03540 = True

IMG_DIR = Path('data/jpg')
labels_raw = scipy.io.loadmat('data/imagelabels.mat')['labels'].flatten()
image_paths = sorted(IMG_DIR.glob('*.jpg'), key=lambda p: int(p.stem.split('_')[1]))
df = pd.DataFrame({'path': image_paths, 'label': labels_raw})

corrupt = []
for p in image_paths:
    try:
        with Image.open(p) as img:
            _ = img.size, img.mode
        corrupt.append(False)
    except (UnidentifiedImageError, OSError):
        corrupt.append(True)
df['corrupt'] = corrupt
df = df[~df['corrupt']].drop(columns=['corrupt'])

if DROP_03540:
    df = df[df['path'].apply(lambda p: p.name) != 'image_03540.jpg']

X = df['path'].values
y = df['label'].values
idx = np.arange(len(df))

X_train_val, X_test, y_train_val, y_test, itr, ite = train_test_split(X, y, idx, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val, itr, iva = train_test_split(X_train_val, y_train_val, itr, test_size=0.15 / 0.85, stratify=y_train_val, random_state=42)

num_classes = len(np.unique(df['label'].values))
label_to_idx = {label: i for i, label in enumerate(sorted(np.unique(df['label'].values)))}

print(f"slika {len(df)} | train {len(X_train)} val {len(X_val)} test {len(X_test)} | klasa {num_classes}")

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
CNN_IMG_SIZE = 128
BATCH_SIZE = 32

class FlowerDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = label_to_idx[self.labels[idx]]
        return img, label

# slike se citaju u originalnoj velicini, kao u main.ipynb, da rezultati ostanu
# uporedivi sa vec izmerenim strategijama
eval_transform = transforms.Compose([
    transforms.Resize((CNN_IMG_SIZE, CNN_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_loader = DataLoader(FlowerDataset(X_val, y_val, transform=eval_transform),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Val batches: {len(val_loader)}")

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, step_counter=None, metrics=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if is_train:
                optimizer.zero_grad()

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            batch_size = imgs.size(0)
            batch_loss = loss.item()
            preds = outputs.argmax(dim=1)
            batch_correct = (preds == labels).sum().item()
            batch_acc = batch_correct / batch_size

            total_loss += batch_loss * batch_size
            total_correct += batch_correct
            total_samples += batch_size

            if is_train and metrics is not None and step_counter is not None:
                step_counter[0] += 1
                metrics["train_loss"].append(batch_loss)
                metrics["train_acc"].append(batch_acc)
                metrics["train_steps"].append(step_counter[0])

    return total_loss / total_samples, total_correct / total_samples


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=100, patience=7, save_path="assets/best_cnn_model.pt"):
    best_val_acc = 0.0
    patience_counter = 0

    metrics = {"train_loss": [], "train_acc": [], "train_steps": [], "val_loss": [], "val_acc": [], "val_steps": []}
    global_step = [0]
    for epoch in range(1, num_epochs + 1):

        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, step_counter=global_step, metrics=metrics)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

        scheduler.step(val_acc)

        metrics["val_loss"].append(val_loss)
        metrics["val_acc"].append(val_acc)
        metrics["val_steps"].append(global_step[0])

        print(f"Epoch {epoch}/{num_epochs}: "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    return best_val_acc, metrics

In [ ]:
class FlowerCNN(nn.Module):
    def __init__(self, num_classes, img_size=CNN_IMG_SIZE, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # img_size / 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # img_size / 4

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # img_size / 8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # img_size / 16
        )

        feat_size = img_size // 16
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * feat_size * feat_size, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# vrednosti iz assets/best_cnn_config.json, iste koje koristi i studija u main.ipynb
best_config = {"lr": 0.0001, "wd": 0.001, "dropout": 0.3}
print(best_config)

In [ ]:
AUG_PATIENCE = 5

# Ista logika kao test_augmentation_configs u main.ipynb, sa jednom razlikom: JSON se
# upisuje posle svake konfiguracije, a konfiguracije koje su vec u fajlu se preskacu.
# Prosli put je Colab reciklirao masinu i rezultati petosatnog prolaza su izgubljeni,
# pa ovako prekid odnese najvise jednu konfiguraciju.
def test_augmentation_configs(AUG_RESULTS_PATH, augmentation_configs, AUG_NUM_EPOCHS = 25):
    if os.path.exists(AUG_RESULTS_PATH):
        with open(AUG_RESULTS_PATH, "r") as f:
            augmentation_results = json.load(f)
        print("Loaded existing augmentation results")
    else:
        augmentation_results = {}

    for aug_name, aug_transform in augmentation_configs.items():
        if aug_name in augmentation_results:
            print(f"{aug_name} -> already done, skipping")
            continue

        started = time.time()
        train_ds_aug = FlowerDataset(X_train, y_train, transform=aug_transform)
        train_loader_aug = DataLoader(train_ds_aug, batch_size=BATCH_SIZE, shuffle=True,
                                      num_workers=2, pin_memory=True)

        aug_model = FlowerCNN(num_classes=num_classes, dropout=best_config["dropout"]).to(device)
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        optimizer = torch.optim.Adam(aug_model.parameters(), lr=best_config["lr"], weight_decay=best_config["wd"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

        best_acc, aug_metrics = train_model(
            aug_model, train_loader_aug, val_loader, criterion, optimizer, scheduler,
            num_epochs=AUG_NUM_EPOCHS, patience=AUG_PATIENCE,
            save_path=f"assets/tmp/aug_{aug_name.replace(' ', '_').replace('+', 'p')}.pt"
        )

        augmentation_results[aug_name] = {
            "best_val_acc": best_acc,
            "val_acc_history": aug_metrics["val_acc"],
        }
        print(f"{aug_name} -> best_val_acc={best_acc:.4f}, {time.time() - started:.0f} s")

        with open(AUG_RESULTS_PATH, "w") as f:
            json.dump(augmentation_results, f, indent=2)

    return augmentation_results

In [ ]:
augmentation_configs_small_rotation = {
    "Flip + Rotation 5": transforms.Compose([
        transforms.Resize((CNN_IMG_SIZE, CNN_IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(5),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    "Flip + Rotation 10": transforms.Compose([
        transforms.Resize((CNN_IMG_SIZE, CNN_IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    "Flip + Rotation 15": transforms.Compose([
        transforms.Resize((CNN_IMG_SIZE, CNN_IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    "Flip + Rotation 15 + ColorJitter": transforms.Compose([
        transforms.Resize((CNN_IMG_SIZE, CNN_IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
}

AUG_RESULTS_PATH_SMALL_ROT = "assets/augmentation_results_small_rotation.json"
AUG_NUM_EPOCHS_SMALL_ROT = 50

started = time.time()
augmentations_results_small_rotation = test_augmentation_configs(
    AUG_RESULTS_PATH_SMALL_ROT, augmentation_configs_small_rotation, AUG_NUM_EPOCHS_SMALL_ROT)
print(f"GOTOVO za {(time.time() - started) / 60:.1f} min")

In [ ]:
# pregled, uz vrednosti koje su vec izmerene na istih 50 epoha
already_measured = {"Flip only": 0.7535, "Flip + ColorJitter": 0.7608,
                    "Flip + ColorJitter + Crop": 0.7714, "Flip + Crop + Affine + RandomCJ": 0.7543}

print("new:")
for name, res in augmentations_results_small_rotation.items():
    print(f"  {name:36s} best_val_acc={res['best_val_acc']:.4f}  epochs={len(res['val_acc_history'])}")
print("existing, 50 epochs, no rotation:")
for name, acc in already_measured.items():
    print(f"  {name:36s} best_val_acc={acc:.4f}")

In [ ]:
# ispis celog JSON-a, da rezultat postoji i u izlazu celije ako preuzimanje padne
print(json.dumps(augmentations_results_small_rotation, indent=2))

In [ ]:
from google.colab import files
files.download(AUG_RESULTS_PATH_SMALL_ROT)